### 1. Setting up spark environment

In [0]:
from pyspark.sql import SparkSession
spark=SparkSession \
    .builder \
        .appName('Databricks_capstone') \
            .getOrCreate()

### 2. configuring storage account

In [0]:
storage_account="mystoacckad"
application_id="14a14259-70ba-4c26-a136-262468ab64da"
directory_id="26af9d76-35fe-404a-b312-869c37aec9c7"
container_name="fileshare"

service_credential = dbutils.secrets.get(scope="Secrete-scope-databricks1", key="app-reg-secrets1")

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
               f"org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", application_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", service_credential)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{directory_id}/oauth2/token")

### Creating dataframes for all tables

In [0]:
folder_path=f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/"
csv=[file_info.name for file_info in dbutils.fs.ls(folder_path) if (file_info.name).endswith('.csv')]

df_customers_dataset=spark.read.csv(folder_path+csv[0],header=True,inferSchema=True)
df_geolocation_dataset=spark.read.csv(folder_path+csv[1],header=True,inferSchema=True)
df_order_items_dataset=spark.read.csv(folder_path+csv[2],header=True,inferSchema=True)
df_order_payments_dataset=spark.read.csv(folder_path+csv[3],header=True,inferSchema=True)
df_order_reviews_dataset=spark.read.csv(folder_path+csv[4],header=True,inferSchema=True)
df_orders_dataset=spark.read.csv(folder_path+csv[5],header=True,inferSchema=True)
df_products_dataset=spark.read.csv(folder_path+csv[6],header=True,inferSchema=True)
df_sellers_dataset=spark.read.csv(folder_path+csv[7],header=True,inferSchema=True)
df_product_category_name_translation=spark.read.csv(folder_path+csv[8],header=True,inferSchema=True)

### Data exploration

In [0]:
# checking Data Lekage or Drop

print(f'Customers : {df_customers_dataset.count()}  rows')
print(f'Geolocation : {df_geolocation_dataset.count()}  rows')
print(f'Order Items : {df_order_items_dataset.count()}  rows')
print(f'Order Payments : {df_order_payments_dataset.count()}  rows')
print(f'Order Reviews : {df_order_reviews_dataset.count()}  rows')
print(f'Orders : {df_orders_dataset.count()}  rows')
print(f'Products : {df_products_dataset.count()}  rows')
print(f'Sellers : {df_sellers_dataset.count()}  rows')
print(f'Product category name translation : {df_product_category_name_translation.count()}  rows')

In [0]:
from pyspark.sql.functions import col, when, count

#check for nulls in critical fields
df_customers_dataset.select([count(when (col(c).isNull(),1)).alias(c) for c in df_customers_dataset.columns]).show()
df_geolocation_dataset.select([count(when(col(c).isNull(),1)).alias(c) for c in df_geolocation_dataset.columns]).show()
df_order_items_dataset.select([count(when(col(c).isNull(),1)).alias(c) for c in df_order_items_dataset.columns]).show()
df_order_payments_dataset.select([count(when(col(c).isNull(),1)).alias(c) for c in df_order_payments_dataset.columns]).show()
df_order_reviews_dataset.select([count(when(col(c).isNull(),1)).alias(c) for c in df_order_reviews_dataset.columns]).show()
df_orders_dataset.select([count(when(col(c).isNull(),1)).alias(c)  for c in df_orders_dataset.columns]).show()
df_products_dataset.select([count(when(col(c).isNull(),1)).alias(c) for c in df_products_dataset.columns]).show()
df_sellers_dataset.select([count(when(col(c).isNull(),1)).alias(c) for c in df_sellers_dataset.columns]).show()
df_product_category_name_translation.select([count(when(col(c).isNull(),1)).alias(c) for c in df_product_category_name_translation.columns]).show()

In [0]:
#duplicate values
df_customers_dataset.groupBy('customer_id').count().withColumnRenamed('count','customer_cnt').filter(col('customer_cnt')>1).show()
df_orders_dataset.groupBy('order_id').count().withColumnRenamed('count','order_id_cnt').filter(col('order_id_cnt')>1).show()
df_sellers_dataset.groupBy('seller_id').count().withColumnRenamed('count','seller_id_cnt').filter(col('seller_id_cnt')>1).show()
df_products_dataset.groupBy('product_id').count().withColumnRenamed('count','product_id_cnt').filter(col('product_id_cnt')>1).show()


In [0]:
#customer distribution by state
df_customers_dataset.groupBy('customer_state').count().orderBy(col('count').desc()).show()

In [0]:
#Order -order status distribution
df_orders_dataset.groupBy('order_status').count().orderBy(col('count').desc()).show()

In [0]:
df_order_payments_dataset.groupBy('payment_type').count().orderBy(col('count').desc()).show()

In [0]:
df_order_items_dataset.show(truncate=False)

In [0]:
#top_selling products

from pyspark.sql.functions import sum
top_product=df_order_items_dataset.groupBy('product_id').count().orderBy(col('count').desc()).show()

In [0]:
#revenue generating products
revenue_gen_products=df_order_items_dataset.groupBy('product_id').agg(sum(col('price')).alias('total_sales_price'))
revenue_gen_products.orderBy(col('total_sales_price').desc()).show()

In [0]:
df_orders_dataset.show()


In [0]:
#average delivery time analysis
delivery_df=df_orders_dataset.select('order_id','order_purchase_timestamp','order_delivered_customer_date', 'order_status')
delivery_df.show()

In [0]:
from pyspark.sql.functions import datediff, to_date

delivery_detail_df=delivery_df \
    .withColumn('delivery_time', datediff(col('order_delivered_customer_date'),col('order_purchase_timestamp'))) \
                .orderBy(col('delivery_time').desc())

delivery_detail_df.show()